# Evaluación del agente — Casos 1 y 2 (Tarea 13)
La lógica de evaluación vive en `src/evaluacion.py` (no se duplica aquí).

In [1]:
import os, sys
from importlib.metadata import version
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))  # raíz del repo (rutas relativas del modelo)
from dotenv import load_dotenv
load_dotenv()
from src.config import GEMINI_MODEL
from src.herramientas import obtener_tools

for pkg in ["langchain", "langgraph", "langchain-google-genai"]:
    print(f"{pkg}: {version(pkg)}")
print("GEMINI_MODEL:", GEMINI_MODEL)
print("GOOGLE_API_KEY definida:", bool(os.getenv("GOOGLE_API_KEY")))
tools = obtener_tools()
for t in tools:
    print(f"\n- {t.name}: {t.description}")

langchain: 1.5.0
langgraph: 1.2.22
langchain-google-genai: 4.5.0
GEMINI_MODEL: gemini-3.5-flash-lite
GOOGLE_API_KEY definida: True

- clasificar_sentimiento_resena: Ejecuta la red neuronal local entrenada con reseñas IMDB en inglés y clasifica su sentimiento.

    Úsala cuando el usuario entregue el texto de una reseña y pida clasificarla o hacer triaje.
    NO la uses para saludos, preguntas sobre políticas ni para un texto ya clasificado en este turno.

    Args:
        texto: la reseña literal y completa, sin resumir ni traducir.

    Returns:
        JSON de una línea con sentimiento ("positivo"/"negativo"), confianza (0-1) y baja_confianza (bool).

- consultar_politica_triaje: Consulta la politica interna y protocolo de respuesta para una resena de cliente.

    Usala cuando conozcas el sentimiento/confianza de la resena o cuando el usuario pregunte
    directamente sobre lineamientos, politicas de triaje o protocolos de escalamiento.
    NO la uses para clasificar texto no anali

In [2]:
from src.agente import construir_agente
from src.evaluacion import (evaluar_caso, imprimir_caso, pausa_entre_casos, guardar_resultados,
                            verificar_caso_1, verificar_caso_2)
agente = construir_agente(tools)
resultados = []
print("Agente construido con", [t.name for t in tools])

Agente construido con ['clasificar_sentimiento_resena', 'consultar_politica_triaje']


## Caso 1
Sin herramientas. Esperadas: `[]`.

In [3]:
r1 = evaluar_caso(agente, "Caso 1", "Hola, ¿qué puedes hacer por mí?", [], verificar_caso_1)
imprimir_caso(r1)
resultados.append(r1)

/Users/steedwar/Documents/GitHub/TallerIA/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 1
Entrada: Hola, ¿qué puedes hacer por mí?
Herramientas esperadas: []
Herramientas invocadas: []
Trayectoria: CUMPLE
Resultado: CUMPLE (respuesta no vacía, sin ToolMessage y sin clasificación inventada)
Respuesta final: Hola. Puedo ayudarte con el triaje automatizado de reseñas de streaming en inglés. 

Mis funciones principales son:
1. **Clasificar el sentimiento** de las reseñas utilizando un modelo neuronal local (positivo o negativo).
2. **Consultar la política interna y el protocolo de respuesta** según el análisis de sentimiento y nivel de confianza.
3. **Determinar la prioridad, la acción recomendada y la pauta** a seguir para cada caso.

¿Tienes alguna reseña o consulta sobre protocolos que desees procesar?
Tokens: 678


## Caso 2
Una sola herramienta. Esperadas: `["consultar_politica_triaje"]`.

In [4]:
print("Pausa según RPM_GRATUITO:", round(pausa_entre_casos()), "s")
r2 = evaluar_caso(agente, "Caso 2",
                  "¿Qué política debo aplicar si un cliente amenaza con acciones legales en su reseña?",
                  ["consultar_politica_triaje"], verificar_caso_2)
imprimir_caso(r2)
resultados.append(r2)

Pausa según RPM_GRATUITO: 5 s


/Users/steedwar/Documents/GitHub/TallerIA/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


/Users/steedwar/Documents/GitHub/TallerIA/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 2
Entrada: ¿Qué política debo aplicar si un cliente amenaza con acciones legales en su reseña?
Herramientas esperadas: ['consultar_politica_triaje']
Herramientas invocadas: ['consultar_politica_triaje']
Trayectoria: CUMPLE
Resultado: CUMPLE (pol_id 'POL_NEG' del ToolMessage aparece en la respuesta)
Respuesta final: - **Sentimiento:** Negativo
- **Confianza:** 1.0
- **Política:** POL_NEG (Prioridad: ALTA)
- **Acción:** Ticket de insatisfacción
- **Pauta:** Empatizar con malestar, agradecer franqueza y ofrecer soporte directo.
Tokens: 1350


## Resumen

In [5]:
import pandas as pd
cumple = lambda ok: "CUMPLE" if ok else "NO CUMPLE"
tabla = pd.DataFrame([{"caso": r["nombre"], "esperadas": r["esperadas"], "invocadas": r["invocadas"],
                       "trayectoria": cumple(r["trayectoria_ok"]), "resultado": cumple(r["resultado_ok"])}
                      for r in resultados])
guardar_resultados(resultados, "assets/apuntes/resultados_casos_1_2.json")
tabla

,caso,esperadas,invocadas,trayectoria,resultado
0,Caso 1,[],[],CUMPLE,CUMPLE
1,Caso 2,[consultar_politica_triaje],[consultar_politica_triaje],CUMPLE,CUMPLE
